In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
print('=' * 60)
print("INICIANDO A TAREFA 1.4 - ANALISE DE DESEMPENHO DO PRODUTO")
print('=' *60)

print("\n Carregando os arquivos necessarios")

In [74]:
df_orders = pd.read_csv(r'C:\GameStoreBrasil\output\orders_cleaned.csv')
print(f' - orders_cleaned.csv :{df_orders.shape[0]} linhas e {df_orders.shape[1]} colunas')

df_games = pd.read_csv(r'C:\GameStoreBrasil\data\games.csv')
print(f' - games.csv :{df_games.shape[0]} linhas e {df_games.shape[1]} colunas')

df_members = pd.read_csv(r'C:\GameStoreBrasil\output\members_cleaned.csv')

 - orders_cleaned.csv :5000 linhas e 7 colunas
 - games.csv :80 linhas e 8 colunas


In [ ]:
print('\n ---- orders_cleaned.csv (primeiras 3 linhas --------)')
print(df_orders.head(3))

print('\n ---- games.csv (primeiras 3 linhas --------)')
print(df_games.head(3))

#Verificar os tipos de dados

print('\n ---- orders_cleaned.csv Tipos de dados')
print(df_orders.dtypes)

print('\n ---- games.csv Tipos de dados')
print(df_games.dtypes)


In [ ]:
#Passo 2: Filtrar anomalias e calcular receita

print('=' * 60)
print("Filtrando Anomalias e Calculando Receita")
print('=' *60)

#1 Quantificar linhas antes do filtro

linhas_antes = len(df_orders)
print(f' -Linhas em orders_cleaned.csv antes do filtro:{linhas_antes}')

In [6]:
#2, Filtrar valores negativos (Regra de Ouro)

df_orders = df_orders[(df_orders['quantity'] >0) & df_orders['unit_price'] > 0]

In [7]:
#3. Quantificar o Impacto do filtro

linhas_depois = len(df_orders)
linhas_removidas = linhas_antes - linhas_depois


In [ ]:
print(f' Linhas depois do filtro {linhas_depois}')
print(f' Linhas removidas  {linhas_removidas}')

In [9]:
#Criar Coluna revenue 

df_orders['revenue'] = df_orders['quantity'] * df_orders['unit_price']

In [ ]:
#Verificação das primerias 5 linhas com a nova coluna 'revenue' 

print("\n Verificação das primerias 5 linhas com a nova coluna 'revenue' ")

print(df_orders.head(5))

In [ ]:
print('=' * 60)

print("Passo 3 - Merge entre Orders e Games ")

print('=' *60)

#Quantificar Linhas antes do merge 
#1. Quantificar linhas antes do merge

linhas_orders_antes = len(df_orders)

print(f' -Linhas em df_orders antes do merge: {linhas_orders_antes}')


In [ ]:
#2 Identificar Games ids orfãos 

game_id_order = set(df_orders['game_id'].unique())
print(game_id_order)
game_ids_games = set(df_games['game_id'].unique())
print(game_ids_games)
game_id_orfaos = game_id_order - game_ids_games
print(game_id_orfaos)

In [ ]:
#Exibir os dados

print(f' Games IDs unicos em orders: {len(game_id_order)}')
print(f' Games IDs unicos em games: {len(game_ids_games)}')
print(f' Games IDs orfãos : {len(game_id_orfaos)}')

In [ ]:
#3 Quantificar quantas linhas serão perdidas com o merge

linhas_com_game_id_orfao = df_orders[df_orders['game_id'].isin(game_id_orfaos)]
linhas_perdidas = len(linhas_com_game_id_orfao)
receita_perdida = linhas_com_game_id_orfao['revenue'].sum()
print(linhas_com_game_id_orfao)

In [ ]:
#4. Fazer o merge (inner join)
# O Merge 'inner' mantém apenas as linhas onde game_id existe em ambas as tabelas
df_merged = df_orders.merge(df_games, on="game_id", how="inner")

#5 Verificar o resultado
linhas_merged = len(df_merged)

print(f'\n linhas apos o merge : {linhas_merged}')

print(f'\n linhas perdidas no merge {linhas_orders_antes - linhas_merged}')


df_merged['Lucro'] = df_merged['revenue'] - (df_merged['cost'] * df_merged['quantity'])
df_perf = df_merged.groupby(["game_id","game_name"]).agg(
 total_quantity = ("quantity","sum"),
 total_revenue=("revenue","sum"),
 lucro = ('Lucro', 'sum')
).reset_index()
print('\n -----------------------------------------------------------')
print(f'Linhas apos a agregação {len(df_perf)}')
print(f'colunas disponiveis {df_perf.shape[1]}')

In [ ]:
#Ordenar por quantidade total ou imprimir sozinho o top 10

print(df_perf['total_quantity'].nlargest(10))

df_perf = df_perf.sort_values('total_quantity', ascending=False)

In [ ]:
#Printando organizado sem nlargest, o sortvalues ja organizou

print("\n --- top 10 jogos por quantidade vendida ---")
print(df_perf[['game_id' , 'game_name', 'total_quantity', 'total_revenue']].head(10))

In [ ]:
print('\n Estatisticas Gerais do Desempenho')
print(f' - Total de jogos únicos com vendas: {len(df_perf)}')
print(f' Quantidade total vendida (todos os jogos) {df_perf['total_quantity'].sum()}')
print(f' Receita total vendida (todos os jogos) {df_perf['total_revenue'].sum()}')
print(f' quantidade Media por jogo (todos os jogos) {df_perf['total_quantity'].mean():.2f}')
print(f' Receita media por jogo (todos os jogos) {df_perf['total_revenue'].mean():.2f}')

In [ ]:
#Agregação de quantidade total de jogo com receita
#Margem de lucro

#2 calcular a margem de lucro absoluta (preco - custo)
df_perf['profit_margin'] = df_perf['price'] - df_perf['cost']

#3. Resultado verificando o top 5 jogos com a maior margem de lucro
df_perf_top_margin = df_perf.sort_values('profit_margin', ascending = False)

print(df_perf.head())

In [ ]:
print("------ top 5 jogos com MAIOR margem de lucro")
print(df_perf_top_margin[['game_name', 'price', 'cost', 'profit_margin']].tail())

In [ ]:
#Receita por plataforma

rev_platform = df_merged.groupby("platform")['revenue'].sum().sort_values(ascending= False) 
print(rev_platform.head(5))

print('\n --- Receita Total por Plataforma ---')
for platforma, revenue in rev_platform.items():
    valor = f"{revenue:,.2f}"
    valor = valor.replace(",", "X").replace(".", ",").replace("X", ".")
    print(f" - {platforma}: R$ {valor}")

In [ ]:
#2. Criar o grafico de barras

fig, ax = plt.subplots(figsize = (8,5))

rev_platform.plot(kind = "bar", ax = ax, color = "#2ca82c", legend = False)

#3. Formatar o grafico

ax.set_title("Receita total por plataforma", fontsize=14, fontweight= "bold", pad = 15)

ax.set_ylabel("receita total (R$)", fontsize = 12)

ax.set_xlabel('Plataforma', fontsize = 12)


In [66]:
#Adicionar os valores em cima de cada barra pra facilitar a leitura

for i, v in enumerate(rev_platform):
    ax.text(i, v + 10000, f"R$ {v/1000:.1f}k", ha= 'center', fontsize = 10, fontweight = "bold")

In [67]:
#Ajustar o layout para nada ser cortado 
plt.tight_layout()

<Figure size 640x480 with 0 Axes>

In [ ]:
#4. Salvar o grafico temporariamente
caminho_grafico_plataforma = r"C:\GameStoreBrasil\output\temp_plataform.png"
plt.savefig(caminho_grafico_plataforma, dpi=150, bbox_inches = 'tight')
print('\n Grafico de barras salvo temporariamente')



In [ ]:
#Salvando e montando o pdf

top3_jogos = df_perf.nlargest(3, 'total_quantity')[['game_name', "total_quantity", "total_revenue"]].copy()

print(top3_jogos.to_string(index=False))

In [ ]:
caminho_pdf = 'output/Session1_ProductPerformance_GameStore.pdf'
print(df_merged)

In [ ]:
with PdfPages(caminho_pdf) as pdf:
    fig1,ax1 = plt.subplots(figsize = (8,5))
    #recalculando a receita por plataforma aqui dentro
    rev_platform = df_merged.groupby("platform")['revenue'].sum().sort_values(ascending = False)
    rev_platform.plot(kind='bar', ax = ax1, color = '#2ca82c', legend = False)
    ax1.set_title('receita total por plataforma', fontsize = 14, fontweight = 'bold', pad = 15)
    ax1.set_ylabel('receita total(R$)', fontsize = 12)
    plt.xticks(rotation = 8)
    for i, v in enumerate(rev_platform):
        ax1.text(i, v + 10000, f"R$ {v/1000:.1f}k", ha= 'center', fontsize = 10, fontweight = "bold")
    plt.tight_layout()
    pdf.savefig(fig1)
    plt.close(fig1)
    fig2, ax2 = plt.subplots(figsize = (8,4))
    dados_tabela = []
    for _, row in top3_jogos.iterrows():
        dados_tabela.append([row['game_name'], int(row['total_quantity']), f"R${row['total_revenue']:,.2f}".replace(',', 'x').replace('.',',').replace('x', '.')])
    #Criar tabela visual
    tabela = ax2.table(celllText = dados_tabela,colLabels=['Jogo', 'qtd.vendida','Receita Total'],cellLoc= 'center', loc = 'center')
    #Titulo da tabela
    ax2.set_Title('top 3 jogos mais vendidos(por quantidade)', fontsize = 14, fontweight = "bold", pad = 20)

    tabela.scale(1.2,1.8)
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(12)

    ax2.set_title("Top 3 jogos Mais vendidos( por quantidade)", fontsize = 14, fontweight = "bold", pad= 20)

    plt.tight_layout()
    pdf.savefig(fig2)
    plt.close(fig2)

    

In [86]:
#Gasto medio por tier

df_gasto_tier = df_members.merge(df_merged, on = 'member_id', how = 'inner')
df_gasto_tier['GastoMedio'] = df_gasto_tier['revenue'].median()
GastoMedioTier = df_gasto_tier.groupby('loyalty_tier')['GastoMedio'].sum().sort_values(ascending= False)
print(GastoMedioTier)

#Agrupar idades por faixa etaria
print(set(df_members['age'].value_counts()))
bins = [0, 18, 30, 45, 60, 100]

labels = [
    "Até 18",
    "19-30",
    "31-45",
    "46-60",
    "60+"
]

df_members["faixa_etaria"] = pd.cut(
    df_members["age"],
    bins=bins,
    labels=labels
)

loyalty_tier
Basic     719213.04
Silver    451374.24
Gold      235054.16
Name: GastoMedio, dtype: float64
{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 22}
